In [61]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision.transforms import ToTensor
from tqdm.auto import tqdm

In [67]:
trainData = datasets.FashionMNIST(train=True, root='data', download=True, transform=ToTensor(), target_transform=None)
# Dont transform the labels of the data
testData = datasets.FashionMNIST(train=False, root='data', download=True, transform=ToTensor())

In [68]:
image, label = trainData[0]

In [69]:
classNames = trainData.classes
print(classNames)

['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


In [70]:
trainDataLoader = DataLoader(trainData, batch_size=32, shuffle=True)
testDataLoader = DataLoader(testData, batch_size=32, shuffle=False)
print(len(trainDataLoader), len(testDataLoader))

1875 313


In [71]:
device = 'cuda' if torch.cuda.is_available() else "cpu"

In [72]:
class FashinMNIST2(nn.Module):
    def __init__(self, inputShape:int, hiddenUnits:int, outputShape:int):
        super().__init__()
        self.Block1 = nn.Sequential(
            nn.Conv2d(in_channels=inputShape, out_channels=hiddenUnits, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hiddenUnits, out_channels=hiddenUnits, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2)
        )
        self.Block2 = nn.Sequential(
            nn.Conv2d(in_channels=hiddenUnits, out_channels=hiddenUnits, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hiddenUnits, out_channels=hiddenUnits, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.Classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=hiddenUnits*7*7, 
                      out_features=outputShape))
    def forward(self, x):
        x = self.Block1(x)
        x = self.Block2(x)
        #print(x.shape)
        x = self.Classifier(x)
        return x

In [73]:
torch.manual_seed(69)
model2 = FashinMNIST2(inputShape=1, hiddenUnits=10, outputShape=len(classNames)).to(device)

In [74]:
print(model2)

FashinMNIST2(
  (Block1): Sequential(
    (0): Conv2d(1, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (Block2): Sequential(
    (0): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (Classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=490, out_features=10, bias=True)
  )
)


In [75]:
lossFunction = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model2.parameters(), lr = 0.01)

In [76]:
def accuracyFunction(yTrue, yPred):
    correct = torch.eq(yTrue, yPred).sum().item()
    acc = (correct/len(yPred)) * 100
    return acc

In [77]:
def printTime(startTime:float, endTime:float, device:torch.device=None):
    time = endTime - startTime
    print(f"Total time on {device}: {time:.3f} seconds")
    return time

In [78]:
def trainStep(dataLoader:torch.utils.data.DataLoader, model:nn.Module,
               optimizer:torch.optim.Optimizer,accuracyFunction, device:torch.device, lossFunction:torch.nn.Module):
    for batch, (X,y) in enumerate(trainDataLoader):
        trainLoss, trainAcc = 0, 0
        X, y = X.to(device), y.to(device)
        yPred = model(X)
        loss = lossFunction(yPred, y)
        trainLoss += loss
        trainAcc += accuracyFunction(yPred=yPred.argmax(dim=1), yTrue=y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    trainLoss /= len(dataLoader)
    trainLoss /= len(dataLoader)
    print(f"Train Loss:{trainLoss:5f} || Train Accuracy: {trainAcc:.2f}\n")
        

In [79]:
def testStep(dataLoader:torch.utils.data.DataLoader, model:nn.Module, device:torch.device, lossFunction:torch.nn.Module, accuracyFunction):
    testLoss, testAcc = 0, 0
    model.to(device)
    model.eval()
    with torch.inference_mode():
        for X, y in dataLoader:
            testPred = model(X)
            loss = lossFunction(testPred, y)
            testLoss += loss
            testAcc += accuracyFunction(yTrue=y, yPred=testPred.argmax(dim=1))
        testLoss /= len(dataLoader)
        testAcc /= len(dataLoader)
        print(f"Test Loss :{testLoss:.5f} || Test Accuracy: {testAcc:.2f}\n")

In [80]:
torch.manual_seed(69)

from timeit import default_timer as timer
trainStartTime = timer()

epochs = 3
for epoch in tqdm(range(epochs)):
    print(f"Epoch {epoch}\n--------------------")
    
    trainStep(dataLoader=trainDataLoader, model=model2, lossFunction=lossFunction, optimizer=optimizer, accuracyFunction=accuracyFunction, device=device)
    testStep(dataLoader=testDataLoader, model=model2, lossFunction=lossFunction, accuracyFunction=accuracyFunction, device=device)

trainEndTime = timer()
totalTrainTime = printTime(startTime=trainStartTime, endTime=trainEndTime, device=device)


  0%|                                              | 0/3 [00:00<?, ?it/s]

Epoch 0
--------------------
Train Loss:0.000000 || Train Accuracy: 78.12



 33%|████████████▋                         | 1/3 [00:43<01:27, 43.86s/it]

Test Loss :0.73186 || Test Accuracy: 71.81

Epoch 1
--------------------
Train Loss:0.000000 || Train Accuracy: 71.88



 67%|█████████████████████████▎            | 2/3 [01:26<00:43, 43.37s/it]

Test Loss :0.55077 || Test Accuracy: 79.58

Epoch 2
--------------------
Train Loss:0.000000 || Train Accuracy: 90.62



100%|██████████████████████████████████████| 3/3 [02:09<00:00, 43.25s/it]

Test Loss :0.42439 || Test Accuracy: 85.04

Total time on cpu: 129.750 seconds


In [81]:
def evalModel(model:nn.Module, dataLoader:torch.utils.data.DataLoader, lossFunction:nn.Module, accuracyFunction):
    loss, acc = 0, 0
    model.eval()
    with torch.inference_mode():
        for X, y in dataLoader:
            yPred = model(X)
            loss += lossFunction(yPred, y)
            acc += accuracyFunction(yTrue=y, yPred=yPred.argmax(dim=1))
        loss /= len(dataLoader)
        acc /= len(dataLoader)
    return {
        "modelName":model.__class__.__name__,
        "modelLoss":loss.item(),
        "modelAcc":acc
    }

In [82]:
model2Results = evalModel(model=model2, dataLoader=testDataLoader, lossFunction=lossFunction, accuracyFunction=accuracyFunction)
print(model2Results)

{'modelName': 'FashinMNIST2', 'modelLoss': 0.4243920147418976, 'modelAcc': 85.04392971246007}
